# Create an `ETC` Reference File using the `RFP`

This notebook will guide users through creating a CRDS format compliant `ETC` Reference file with the RFP. 

-----

## Setup

In order to run this notebook and generate the Mask reference file, a `conda` environment must be created that has the `roman-wfi-reference-file-pipeline` code installed. 

To download `conda` on your machine, please follow the installation instructions on the [`conda` website](https://docs.conda.io/projects/conda/en/latest/user-guide/install/index.html). 

Once `conda` is installed, go to [the RFP's Github page](https://github.com/spacetelescope/roman-wfi-reference-pipeline) and follow [the RFP installation instructions](https://github.com/spacetelescope/roman-wfi-reference-pipeline#installation).


Make sure that the version of [roman_datamodels](https://github.com/spacetelescope/roman_datamodels) is 0.31.0 or higher for the ETC reference file. 

When the code is properly installed, users can run the notebook examples.

-----

## Imports

A few packages are needed when running this notebook. We will import them in the cell below:

In [ ]:
# Used to generate the reference file (required!)
# Used to easily retrieve a filelist
import glob
import os

import asdf
import matplotlib.pyplot as plt

# Used to plot and create fake data for this notebook, optional
import numpy as np
from astropy.io import fits


from wfi_reference_pipeline.reference_types.exposure_time_calculator.exposure_time_calculator import ExposureTimeCalculator, update_etc_form_from_crds
from wfi_reference_pipeline.resources.make_dev_meta import MakeDevMeta

## About the RFP `ETC` Module

The `ETC` module is designed to gather detector parameters for `pandeia`, which is the engine for [the Roman Wide Field Instrument (WFI) Exposure Time Calculator (ETC)](https://roman.etc.stsci.edu). The module queries the reference files on [CRDS](https://roman-crds.stsci.edu/), stores the values in a yaml file, builds structured data containing the parameters and metadata, and then saves it in [ASDF](https://roman-docs.stsci.edu/data-handbook/wfi-data-levels-and-products/introduction-to-asdf) format. All values in the output ASDF files are either in electrons or electrons per second. 

The module is consisted of 2 parts: the `ETC` class and a standalone function called `update_etc_form_from_crds`. `update_etc_form_from_crds` is used to update a yaml file (exposure_time_calculator_form.yml) located in '~/roman-wfi-reference-pipeline/src/wfi_reference_pipeline/reference_types/exposure_time_calculator/' with the downloaded reference files for the detector parameters from CRDS. 

The `ETC` class inherits from the `ReferenceType` base class. There are no user-required inputs for the `ETC` object, but the class reads in the updated yaml file from `update_etc_form_from_crds`. The `ETC` class populates metadata compatible with the Roman datamodel, builds the Roman datamodel tree by combining the metadata and the detector parameters, and returns an output file in ASDF format, which is CRDS-compatible. 

Each part of the `ETC` module can be run asynchronously. Here, we explain how to run each part of the module.

-----
### Part 1: Update exposure_time_calculator_form.yml using `update_etc_form_from_crds`

The code snippet below shows you how to update the yaml file using the `ETC` module's standalone function `update_etc_form_from_crds`. The function queries the CRDS server for reference files, stores them locally, and updates the yaml form with predetermined metrics for readnoise, dark current, flat field, and saturation values. All reference files are in instrumental units of data numbers (DN) but the ETC operates in electrons. `update_etc_form_from_crds` handles the gain correction and converts DNs into electrons. 

In [ ]:
# Set the path where the files are cached
outpath = '/Path/to/desired/location/'

# Call the function to update the yaml file with the latest reference files
update_etc_form_from_crds(outpath)

Now that we have updated the existing yml file, we can move on to write the parameters into the asdf format. 

-----
### Part 2: Create ASDF files for each detector

The code snippet below shows how to save the parameters in the yaml form into the CRDS-compatible ASDF format using the `ETC` class in the module. We will loop through all 18 detectors and generate 18 ASDF files, one for each detector. 

We will first create a `MakeDevMeta` object and update some metadata. These metadata will be used in the CRDS submission. 

In [ ]:
# Corresponding ETC release to deliver the reference files
etc_release = 'R2027.3'

tmp = MakeDevMeta(ref_type="ETC")
print("The default metadata values are: ", tmp.meta_etc)

# Manually setting various metadata
tmp.meta_etc.use_after = '2020-05-01T00:00:00.000'
tmp.meta_etc.author = "Eunkyu Han"
tmp.meta_etc.description = f"Creating an ETC reference file for {etc_release} release"

print("The new metadata values are: ", tmp.meta_etc)

Now we can create the `ETC` object. We will set meta_data to `tmp.meta_etc`. We will set `file_list` to `None`, since we will use the form we updated in the `ETC` module. We will set `ref_type_data` to `None`. We will also specify an output filename for the reference file (we will use `roman_etc_file_wfi##.asdf` where ## corresponds to the detector number) and we will set `clobber` to `True` so we can overwrite existing files.

Once we successfully create the `ETC` object, we can generate the reference files using the function `generate_outfile()`. We will loop through all 18 detectors and create 18 output files. 

In [ ]:
# Loop through all 18 detectors to output the ASDF files
for wfi_num in range(18):
    detector = wfi_num + 1
    # Update the detector number in the metadata
    tmp.meta_etc.instrument_detector = f"WFI{detector:02d}"

    # Creating the ETC object
    rfp_etc = ExposureTimeCalculator(meta_data=tmp.meta_etc,
                                     file_list=None,
                                     outfile=f"roman_etc_file_wfi{detector:02}.asdf",
                                     clobber=True)
    
    # Generating the CRDS-compliant reference file
    rfp_etc.generate_outfile()

Let's check one of the output files to verify if everything looks good. 

In [ ]:
# Check the file
with asdf.open(rfp_etc.outfile) as af:

    etc = af["roman"]

## Optional part
The part below shows you how to update the exposure_time_calculator_form.yml once again using the `add_eol_comments_for_wfi01` function. Preferably, this part is done after you create the ASDF files to add notes about what each detector parameter means in the ETC engine. They are added as comments to the WFI01 properties.

In [ ]:
from wfi_reference_pipeline.reference_types.exposure_time_calculator.exposure_time_calculator import add_eol_comments_for_wfi01

yaml_file = "~/roman-wfi-reference-pipeline/src/wfi_reference_pipeline/reference_types/exposure_time_calculator/exposure_time_calculator_form.yml"
add_eol_comments_for_wfi01(yaml_file, 
                 {'saturaion_on': 'Saturation used in ETC',
                  'saturation_fullwell': 'Typical fullwell depth. Used to determine the saturation by the ETC',
                  'dark_current_on': 'Dark used in ETC',
                  'dark_current': 'Median dark value in electrons/s',
                  'readnoise_on': 'Read noise used in ETC',
                  'readnoise': 'Median read noise value in electrons',
                  'flat_field_electrons': 'Total number of electrons in the superflat. Used to determine the flat fielding error by the ETC'}
                 )

-----

This notebook was written by Eunkyu Han on June 10, 2026.